# Base de Dados Escolhida para Estudo:
Ocorrências Aeronáuticas - ANAC
https://dados.gov.br/dados/conjuntos-dados/ocorrencias-aeronauticas

# Metadados:
https://www.anac.gov.br/acesso-a-informacao/dados-abertos/areas-de-atuacao/seguranca-operacional/ocorrencias-aeronauticas/76-ocorrencias-aeronauticas

# Dados Auxiliares
PDF: MCA 3-6 - Manual de Investigação do SIPAER
https://www2.fab.mil.br/cenipa/index.php/anexos/24-legsipaer

IDH:


# Quais as Perguntas  
1- Quais os 3 tipos mais comuns de ocorrência por estado?  
2- Qual a média de ocorrências por região por ano?  
3- Quais fases das operações geram mais acidentes?  
4- Qual o total de fatalidades por operação por ano?  
5- Qual o tipo mais comum de ocorrência por categoria da aeronave?  
6- Qual a categoria de aeronave com menor indice de fatalidades? (total de fatalidades / total de ocorrências / número de acentos*)  
*normalização para comparação com demais aeronaves

# Importando Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import requests
from io import StringIO

# Importando a base de dados do GitHub

In [ ]:
def carregar_csv_github(url_raw: str) -> pd.DataFrame:
    """
    Baixa um CSV público do GitHub e retorna como DataFrame pandas.
    url_raw: Caminho no GitHub para o arquivo .CSV
    """
    response = requests.get(url_raw, timeout=30)
    response.raise_for_status()
    
    # 1. Lê o arquivo pulando os metadados do topo
    df_lido = pd.read_csv(
        StringIO(response.text), 
        sep=';', 
        skiprows=1, 
        on_bad_lines='skip'
    )
    
    # 2. Transforma a string textual 'null' no valor NaN real do sistema
    df_limpo = df_lido.replace('null', np.nan)
    
    # 3. Retorna o DataFrame perfeitamente estruturado
    return df_limpo

# Execução do código
csv_path = "https://raw.githubusercontent.com/Jobanu/Projeto-Final-DS-PY-004/main/Dados/V_OCORRENCIA_AMPLA.csv"
df = carregar_csv_github(csv_path)

print(f"Carregadas {len(df)} ocorrências")
df.head()


In [ ]:
tamanho_df_inicio = len(df)
print(f"Tamanho inicial do DataFrame: {tamanho_df_inicio}")
memoria_df_inicio = df.memory_usage(deep=True).sum()
print(f"Tamanho inicial do DataFrame bytes: {memoria_df_inicio}")

# Explorando a base de dados

In [ ]:
# Dimensões
df.shape

In [ ]:
# Visualizando as 5 primeiras linhas
df.head()

In [ ]:
# Visualizando as 5 últimas linhas
df.tail()

In [ ]:
# Visualizando os Metadados do Dataframe
df.info()

In [ ]:
print(df.isna().sum())

Atributos de [Data_da_Ocorrencia] e [Hora_da_Ocorrencia] carregados como object, é necessário ajustar este carregamento no comando de leitura do .csv:  
 4   Data_da_Ocorrencia                5021 non-null   object   
 5   Hora_da_Ocorrencia                3588 non-null   object  

 45 Colunas disponíveis. Será necessário selecionar as colunas próprias para o estudo, uma vez que fica inviável tratar todos dados nulls, vazios, outliners e depois não usar esses dados para o estudo.  
   

Será Necessário tratamento com cruzamento deste dados para preencimento do que for possível:  
Municipio                           1394  
UF                                     3  
Regiao                               670  

In [ ]:
df[df['UF'].isnull()][['UF', 'Historico']]

É possível inferir que o atributo UF do ID 4705 é GO, os demais podem ser excluídos para o estudo de acidentes no território nacional, uma vez que, pelo histórico, ocorreram fora do território nacional.

In [ ]:
df[df['Regiao'].isnull()][['UF', 'Historico']]

Podemos Excluir os registros com atributos [UF] == Exterior, uma vez que se trata de ocorrências fora do território nacional e, portanto, fora do range deste estudo que se restringe as ocorrências em território nacional.  
Podemos Excluir os registros com atributos [UF] == Indeterminado e [Histórico] == NaN, pois não dá para determinar o Estado da ocorrência para o estudo.  
Os demais Indeterminados, podem ser inferidos pelo histórico, mas devemos considerar o custo de uma Machine Learning para extrair esta informação ou fazer manualmente um a um.

Após o tratamento, devemos fazer a correlação [UF] x [Regiao] para preencher os dados faltantes de [Regiao] atráves dos dados existentes de [UF]

# Filtros do Estudo 
1 - [Numero_da_Ocorrencia] (tratando como indice)  
2 - [Classificacao_da_Ocorrencia]? 'Acidente' e/ou 'Incidente Grave'  
3 - Período do estudo baseado em [Data_da_Ocorrencia]  (Quais anos?)  
# Atributos de Interesse para as perguntas
1- agrupar por([UF]); contar([Tipo_de_Ocorrencia]); <-corresponde([Descricao_do_Tipo]); filtra(ano([Data_da_Ocorrencia]))  
  
2- agrupar por [Regiao]; agrupar por ano([Data_da_Ocorrencia]); média(soma([Lesoes_Fatais_Tripulantes]) soma([Lesoes_Fatais_Passageiros]) soma([Lesoes_Fatais_Terceiros]))  
  
3- agrupar por [Fase_da_Operacao]; contar([Numero_da_Ocorrencia])  
  
4- agrupar por [Fase_da_Operacao]; soma(soma([Lesoes_Fatais_Tripulantes]) soma([Lesoes_Fatais_Passageiros]) soma([Lesoes_Fatais_Terceiros])  
  
5- agrupar por [Categoria_da_Aeronave]; agrupar por [Tipo_de_Ocorrencia]. contar([Numero_da_Ocorrencia])  

6- agrupar por [Categoria_da_Aeronave]; 	soma(soma([Lesoes_Fatais_Tripulantes]) soma([Lesoes_Fatais_Passageiros]) soma([Lesoes_Fatais_Terceiros]) /
										contagem([Numero_da_Ocorrencia]) /
										soma([Numero_de_Assentos])


In [ ]:
Atritubos_interesse = [
    'Numero_da_Ocorrencia',
    'Classificacao_da_Ocorrencia',
    'Data_da_Ocorrencia',
    'Historico',
    'UF',
    'Regiao',
    'Tipo_de_Ocorrencia',
    'Descricao_do_Tipo',
    'Operacao',
    'Lesoes_Fatais_Tripulantes',
    'Lesoes_Fatais_Passageiros',
    'Lesoes_Fatais_Terceiros',
    'Lesoes_Graves_Tripulantes',
    'Lesoes_Graves_Passageiros',
    'Lesoes_Graves_Terceiros',
    'Lesoes_Leves_Tripulantes',
    'Lesoes_Leves_Passageiros',
    'Lesoes_Leves_Terceiros',
    'Ilesos_Tripulantes',
    'Ilesos_Passageiros',
    'Lesoes_Desconhecidas_Tripulantes',
    'Lesoes_Desconhecidas_Passageiros',
    'Lesoes_Desconhecidas_Terceiros',
    'Fase_da_Operacao',
    'Categoria_da_Aeronave',
    'Numero_de_Assentos'
    ]

Será necessário estudar o forma de preencher os dados Nulls nestes Atributos, pois são atributos de interesse no estudo:  
Lesoes_Fatais_Tripulantes           1118  
Lesoes_Fatais_Passageiros           1239  
Lesoes_Fatais_Terceiros             1319  
Lesoes_Graves_Tripulantes           1168  
Lesoes_Graves_Passageiros           1281  
Lesoes_Graves_Terceiros             1327  
Lesoes_Leves_Tripulantes            1124  
Lesoes_Leves_Passageiros            1277  
Lesoes_Leves_Terceiros              1327  
Ilesos_Tripulantes                   410  
Ilesos_Passageiros                   949  
Lesoes_Desconhecidas_Tripulantes    1315  
Lesoes_Desconhecidas_Passageiros    1326  
Lesoes_Desconhecidas_Terceiros      1327  

In [ ]:
# 1. Lista com as colunas de lesões e contagem da base
colunas_lesoes = [
    'Lesoes_Fatais_Tripulantes', 'Lesoes_Fatais_Passageiros', 'Lesoes_Fatais_Terceiros',
    'Lesoes_Graves_Tripulantes', 'Lesoes_Graves_Passageiros', 'Lesoes_Graves_Terceiros',
    'Lesoes_Leves_Tripulantes', 'Lesoes_Leves_Passageiros', 'Lesoes_Leves_Terceiros',
    'Ilesos_Tripulantes', 'Ilesos_Passageiros',
    'Lesoes_Desconhecidas_Tripulantes', 'Lesoes_Desconhecidas_Passageiros', 'Lesoes_Desconhecidas_Terceiros'
]

df[colunas_lesoes].head(10)

# Relendo o CSV com os novos filtros

In [ ]:
# Importando a base de dados do GitHub com filtros
def carregar_csv_github2(url_raw: str, pula_linha = 1) -> pd.DataFrame:
    """
    Baixa um CSV público do GitHub e retorna como DataFrame pandas.
    url_raw: Caminho no GitHub para o arquivo .CSV
    """
    response = requests.get(url_raw, timeout=30)
    response.raise_for_status()

    # 1. Lê o arquivo agora com somente os atributos de interesse
    # e com configuração de correta da [Data_da_Ocorrencia]
    df_lido = pd.read_csv(
        StringIO(response.text),
        sep=';',
        skiprows=pula_linha,
        on_bad_lines='skip',
        usecols=Atritubos_interesse,
        parse_dates=['Data_da_Ocorrencia']
    )

    # 2. Transforma a string textual 'null' no valor NaN real do sistema
    df_limpo = df_lido.replace('null', np.nan)

    # 3. Retorna o DataFrame perfeitamente estruturado
    return df_limpo

# Execução do código
df = carregar_csv_github2(csv_path)

print(f"Carregadas {len(df)} ocorrências")

# Verificando as novas dimensões do Dataframe

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
print(df.isna().sum())

# Iniciando o tratamento dos Dados

## Lesões

## Tratamento das colunas dos tipos de Lesões - (NaN para 0)

A razão para escolhermos apenas as colunas de lesões e não o DataFrame inteiro é que o significado do "vazio" (NaN) muda completamente dependendo do tipo de dado da coluna. Substituir tudo por zero de forma automática geraria graves erros de lógica e distorções na análise.

1. Colunas de Texto (Strings) ficariam sem sentido

    - Operador: O responsável pelo voo passaria a se chamar 0 (o Pandas não saberia que ali deveria ser "Desconhecido")
    Matricula: O prefixo do avião viraria 0.

2. Colunas Numéricas Específicas seriam corrompidas, pois onúmero 0 possui um valor matemático real em engenharia aeronáutica.

    - PMD (Peso Máximo de Decolagem): Se uma aeronave experimental não teve o peso registrado, colocar 0 faria o Pandas calcular que o avião pesa zero quilos, o que estragaria qualquer média de peso que você tentasse fazer no projeto.
    - Numero_de_Assentos: Colocar 0 indicaria que o avião voava sem nenhum banco dentro, quando na verdade o dado apenas não foi coletado.

3. Nas colunas de Lesões, o 0 mantém a lógica perfeitaNas colunas de contagem de pessoas (Lesoes_Fatais, Lesoes_Graves), assumir que o vazio equivale a 0 é seguro e correto para o modelo de dados, pois:

    - Se o relatório não mencionou feridos, a hipótese estatística padrão é que zero pessoas se machucaram naquela categoria específica.
    - Permite converter a coluna de Float (número quebrado) para Int (número inteiro), já que não existem "1.5 pessoas mortas".

In [ ]:
# 1. Lista com as colunas de lesões e contagem da base
colunas_lesoes = [
    'Lesoes_Fatais_Tripulantes', 'Lesoes_Fatais_Passageiros', 'Lesoes_Fatais_Terceiros',
    'Lesoes_Graves_Tripulantes', 'Lesoes_Graves_Passageiros', 'Lesoes_Graves_Terceiros',
    'Lesoes_Leves_Tripulantes', 'Lesoes_Leves_Passageiros', 'Lesoes_Leves_Terceiros',
    'Ilesos_Tripulantes', 'Ilesos_Passageiros',
    'Lesoes_Desconhecidas_Tripulantes', 'Lesoes_Desconhecidas_Passageiros', 'Lesoes_Desconhecidas_Terceiros'
]

# --- ANTES ---
print("=== 1. QUANTIDADE DE NaNs ANTES DA LIMPEZA ===")
# Soma quantos NaNs existem apenas nas colunas da lista
print(df[colunas_lesoes].isna().sum())
print("-" * 50)


# 2. Criar a função de preenchimento
def preencher_nan_lesoes(dataframe: pd.DataFrame, lista_colunas: list) -> pd.DataFrame:
    """
    Substitui os valores NaN das colunas de lesões pelo número zero (0).
    Aplica a conversão para tipo inteiro (Int64), ideal para contagens.
    """
    # Faz uma cópia para não alterar o DataFrame original por acidente
    df_copia = dataframe.copy()
    
    # Preenche os NaNs com 0 e converte para tipo numérico inteiro
    df_copia[lista_colunas] = df_copia[lista_colunas].fillna(0).astype('int64')
    
    return df_copia


# 3. Aplicar a função no seu DataFrame
df_tratado = preencher_nan_lesoes(df, colunas_lesoes)


# --- DEPOIS ---
print("=== 2. QUANTIDADE DE NaNs DEPOIS DA LIMPEZA ===")
print(df_tratado[colunas_lesoes].isna().sum())
print("-" * 50)



## Valores únicos

In [ ]:
# Lista para armazenar os dados dos valores únicos de cada coluna
dados_valores_unicos = []

# Itera sobre cada coluna do DataFrame df_tratado
for coluna in df_tratado.columns:
    valor_unico = df_tratado[coluna].unique()
    numero_valores_unicos = len(valor_unico)
    dados_valores_unicos.append({
        'Coluna': coluna,
        'Valores_Unicos': valor_unico.tolist(), # Converte array numpy para lista
        'Numero_de_Valores_Unicos': numero_valores_unicos
    })

# Cria um DataFrame a partir dos dados coletados
df_valores_unicos = pd.DataFrame(dados_valores_unicos)

print('==== Resumo de Valores Únicos por Atributo no df_tratado ====\n')
# Exibe o DataFrame resultante
display(df_valores_unicos)

## [Numero_de_Assentos]

In [ ]:
df['Numero_de_Assentos'].isna().sum()

Trantando o [Numero_de_assentos] pela média existente calculada pela [Categoria_da_Aeronave]

In [ ]:
media_assentos_por_categoria = df_tratado.groupby('Categoria_da_Aeronave')['Numero_de_Assentos'].mean().reset_index()
media_assentos_por_categoria = media_assentos_por_categoria.dropna(subset=['Numero_de_Assentos'])
media_assentos_por_categoria['Numero_de_Assentos'] = media_assentos_por_categoria['Numero_de_Assentos'].astype('int64')
display(media_assentos_por_categoria)

In [ ]:
# Criar um dicionário de mapeamento de Categoria_da_Aeronave para Numero_de_Assentos médios
assentos_medios_dict = media_assentos_por_categoria.set_index('Categoria_da_Aeronave')['Numero_de_Assentos'].to_dict()

# Mapear os valores médios para a coluna 'Numero_de_Assentos' do df_tratado
# Usar apply com uma função lambda para preencher apenas NaNs
df_tratado['Numero_de_Assentos'] = df_tratado.apply(
    lambda row: assentos_medios_dict.get(row['Categoria_da_Aeronave'], row['Numero_de_Assentos'])
    if pd.isna(row['Numero_de_Assentos']) else row['Numero_de_Assentos'],
    axis=1
)

# Converter a coluna para int64 após o preenchimento, se necessário (lidando com NaNs remanescentes, se houver)
df_tratado['Numero_de_Assentos'] = df_tratado['Numero_de_Assentos'].fillna(0).astype('int64')

print(f"NaNs restantes em 'Numero_de_Assentos': {df_tratado['Numero_de_Assentos'].isnull().sum()}")
display(df_tratado[['Categoria_da_Aeronave', 'Numero_de_Assentos']].head())

## [UF] e [Região]

### [UF]

In [ ]:
df_tratado['UF'].isna().sum()

In [ ]:
df[df['UF'].isnull()][['Numero_da_Ocorrencia', 'UF', 'Historico']]

In [ ]:
#Conforme Histórico, ID 4705 tem a UF em GO
df_tratado.loc[df_tratado['Numero_da_Ocorrencia'] == 46147, 'UF'] = 'GO'
display(df_tratado[df_tratado['UF'].isnull()][['Numero_da_Ocorrencia', 'UF', 'Historico']])

Com o [Numero_da_Ocorrencia] 46147 já tratado, vamos agora remover as ocorrências onde a [UF] ainda está como nula. Conforme a análise anterior, estas são ocorrências fora do território nacional e não são relevantes para o estudo.

In [ ]:
df_tratado.dropna(subset=['UF'], inplace=True)
print(f"NaNs restantes em 'UF': {df_tratado['UF'].isnull().sum()}")

### [Regiao]

Vamos preencher os valores ausentes no atributo [Regiao] do df_tratado. Para isso, usaremos o dicionário regioes_brasil para mapear cada [UF] à sua Regiao correspondente.

In [ ]:
# Dicionário de Regiões do Brasil
regioes_brasil = {
    "Norte": ["AC", "AM", "AP", "PA", "RO", "RR", "TO"],
    "Nordeste": ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"],
    "Centro-Oeste": ["DF", "GO", "MT", "MS"],
    "Sudeste": ["ES", "MG", "RJ", "SP"],
    "Sul": ["PR", "RS", "SC"]
}


In [ ]:
# Inverter o dicionário para mapear UF para Região
uf_para_regiao = {
    uf: regiao
    for regiao, ufs in regioes_brasil.items()
    for uf in ufs
}

# Criar um Series com as novas regiões mapeadas a partir da UF
# Valores de UF que não estão no dicionário resultarão em NaN aqui
regioes_mapeadas = df_tratado['UF'].map(uf_para_regiao)

# Preencher APENAS os NaN em 'Regiao' com os valores de 'regioes_mapeadas'
# O aviso FuturegWarning é resolvido ao atribuir o resultado de volta à coluna
df_tratado['Regiao'] = df_tratado['Regiao'].fillna(regioes_mapeadas)

print(f"NaNs restantes em 'Regiao': {df_tratado['Regiao'].isnull().sum()}")

Vamos investigar quais valores de [UF] não foram mapeados e ainda resultam em NaN na coluna [Regiao].

In [ ]:
ufs_nao_mapeadas = df_tratado[df_tratado['Regiao'].isnull()]['UF'].unique()
print(f"UFs que não foram mapeadas para 'Regiao': {ufs_nao_mapeadas.tolist()}")

Vamos proceder à remoção dos registros contendo 'Exterior', pois não faram parte do estudo

In [ ]:
# Remove as linhas onde 'UF' == 'Exterior'
df_tratado = df_tratado[~df_tratado['UF'].isin(['Exterior'])]

# Verifica novamente os NaNs na coluna 'Regiao' após a remoção
print(f"NaNs restantes em 'Regiao' após a remoção de 'Exterior': {df_tratado['Regiao'].isnull().sum()}")

Vamos remover os registros onde a [UF] é 'Indeterminado' e o [Historico] está NaN.

In [ ]:
# Remove as linhas onde 'UF' é 'Indeterminado' E 'Historico' é NaN
df_tratado = df_tratado[~((df_tratado['UF'] == 'Indeterminado') & (df_tratado['Historico'].isnull()))]

# Verifica novamente os NaNs na coluna 'Regiao' após a remoção e re-mapeamento
print(f"NaNs restantes em 'Regiao' após a remoção de 'Indeterminado' com Historico NaN: {df_tratado['Regiao'].isnull().sum()}")

Como não está no escopo do estudo a utilização de Machine Learning, IA Generativa ou Agentes de IA para tratamento dos dados, usando o [Historico] como dado para inferencia da [UF], vamos excluir os demais Indeterminados para continuação do estudo

In [ ]:
# Remove as linhas onde 'UF' == 'Indeterminado'
df_tratado = df_tratado[~df_tratado['UF'].isin(['Indeterminado'])]

# Verifica novamente os NaNs na coluna 'Regiao' após a remoção
print(f"NaNs restantes em 'Regiao' após a remoção de 'Indeterminado': {df_tratado['Regiao'].isnull().sum()}")

## [Fase_da_Operacao]

In [ ]:
display(df_tratado[df_tratado['Fase_da_Operacao'].isnull()][['Fase_da_Operacao', 'Descricao_do_Tipo', 'Historico']])

Não é possível inferir uma [Fase_da_Operação] a partir dos dados que temos para preenchimento dos registros com NaN, portanto, optamos por excluir do estudo estes registros.

## [Tipo_de_Ocorrencia]

In [ ]:
df_tp_null = df_tratado[df_tratado['Tipo_de_Ocorrencia'].isnull()][['Numero_da_Ocorrencia', 'Tipo_de_Ocorrencia', 'Descricao_do_Tipo', 'Historico']]
df_tp_null.to_csv('df_tp_null.csv', index=False)
df_tp_null

Como temos 20 ocorrências, vamos analisar e atualizar manualmente os dados de tipo de ocorrência, de acordo com a tabela de Tipos de Ocorrências do 'MCA 3-6 - Manual de Investigação do SIPAER' - Anexo C, pags. 359 - 380.  
Faremos os acertos usando o df_tp_null.csv criado, retornando com um o tp_null_acertado.csv com os valores a serem corrigidos.


In [ ]:
# Importando a base de dados do GitHub
def carregar_csv_github(url_raw: str, pula_linha = 1) -> pd.DataFrame:
    """
    Baixa um CSV público do GitHub e retorna como DataFrame pandas.
    url_raw: Caminho no GitHub para o arquivo .CSV
    """
    response = requests.get(url_raw, timeout=30)
    response.raise_for_status()

    # 1. Lê o arquivo pulando os metadados do topo
    df_lido = pd.read_csv(
        StringIO(response.text),
        sep=',',
        on_bad_lines='skip'
    )

    # 2. Transforma a string textual 'null' no valor NaN real do sistema
    df_limpo = df_lido.replace('null', np.nan)

    # 3. Retorna o DataFrame perfeitamente estruturado
    return df_limpo

# Execução do código
tp_null_path = "https://raw.githubusercontent.com/Jobanu/Projeto-Final-DS-PY-004/main/Dados/tp_null_acertado.csv"
df_tp = carregar_csv_github(tp_null_path)

print(f"Carregadas {len(df_tp)} ocorrências")


Com base nos dados corrigidos do df_tp contendo os acertos manuais, vamos atualizar as colunas [Tipo_de_Ocorrencia] e [Descricao_do_Tipo] no df_tratado para preencher os NaN's.

In [ ]:
# Cria dicionários de mapeamento a partir de df_tp
tipo_ocorrencia_map = df_tp.set_index('Numero_da_Ocorrencia')['Tipo_de_Ocorrencia'].to_dict()
descricao_tipo_map = df_tp.set_index('Numero_da_Ocorrencia')['Descricao_do_Tipo'].to_dict()

# Identifica as linhas em df_tratado onde 'Tipo_de_Ocorrencia' é NaN
mask_tipo_nan = df_tratado['Tipo_de_Ocorrencia'].isnull()
# Identifica as linhas em df_tratado onde 'Descricao_do_Tipo' é NaN
mask_descricao_nan = df_tratado['Descricao_do_Tipo'].isnull()

# Preenche os NaNs em 'Tipo_de_Ocorrencia' usando o mapeamento
df_tratado.loc[mask_tipo_nan, 'Tipo_de_Ocorrencia'] = df_tratado.loc[mask_tipo_nan, 'Numero_da_Ocorrencia'].map(tipo_ocorrencia_map)

# Preenche os NaNs em 'Descricao_do_Tipo' usando o mapeamento
df_tratado.loc[mask_descricao_nan, 'Descricao_do_Tipo'] = df_tratado.loc[mask_descricao_nan, 'Numero_da_Ocorrencia'].map(descricao_tipo_map)

# Verifica quantos NaNs ainda existem na coluna 'Tipo_de_Ocorrencia'
remaining_nan_tipo = df_tratado['Tipo_de_Ocorrencia'].isnull().sum()
print(f"NaNs restantes em 'Tipo_de_Ocorrencia' após a atualização: {remaining_nan_tipo}")

As demais ocorrências seram apagadas, pois não é possível determinar estes dados.

In [ ]:
df_tratado.dropna(subset=['Tipo_de_Ocorrencia'], inplace=True)
print(f"NaNs restantes em 'Tipo_de_Ocorrencia': {df_tratado['Tipo_de_Ocorrencia'].isnull().sum()}")

### [Categoria_da_Aeronave]

In [ ]:
df_tratado[df_tratado['Categoria_da_Aeronave'].isnull()][['Numero_da_Ocorrencia', 'Categoria_da_Aeronave', 'Historico']]

Não é possível definir a [Categoria_da_Aeronave] pelo [Historico], portanto, vamos excluir as linhas não preenchidas.


In [ ]:
df_tratado.dropna(subset=['Categoria_da_Aeronave'], inplace=True)
print(f"NaNs restantes em 'Categoria_da_Aeronave': {df_tratado['Categoria_da_Aeronave'].isnull().sum()}")

### Consultando as informações do DataFrame para verificação do sucesso da limpeza de NaN's

In [ ]:
df_tratado.info()

### [Numero_da_Ocorrencia]

Ocorrências duplicadas

In [ ]:

# Calcula as ocorrencias em 'Numero_da_Ocorrencia'
conta_ocorrencias = df_tratado['Numero_da_Ocorrencia'].value_counts()

# Filtra valores que aparecem mais de uma vez
ocorrencias_repetidas = conta_ocorrencias[conta_ocorrencias > 1]

# Converte o resultado para NumPy array
ocorrencias_repetidas_array = np.array([
    [num_oc, contagem] for num_oc, contagem in ocorrencias_repetidas.items()
])

print("Valores repetidos no atributo 'Numero_da_Ocorrencia' e suas contagens:")
print(ocorrencias_repetidas_array)

Verificando a relação de tipos de Ocorrências nos Números de ocorrências repetidos.  
Queremos saber se as ocorrências repetidas estão relacionanda a tipos de ocorrências diferentes, ou seja, possíveis agentes diferentes para mesma ocorrência ou erro de registro realmente duplicado.

In [ ]:
# Extrai os números de ocorrência que se repetem, a partir da análise
# anterior
numeros_ocorrencias_duplicadas = ocorrencias_repetidas.index.tolist()

# Lista para armazenar os resultados
lista_resultados_tipo_ocorrencia = []

# Itera sobre cada número de ocorrência que foi detectado como duplicado
for num_ocorrencia in numeros_ocorrencias_duplicadas:
    # Filtra o DataFrame original (df_tratado) para este número de ocorrência
    df_filtrado_por_num = df_tratado[df_tratado['Numero_da_Ocorrencia'] == num_ocorrencia]

    # Coleta todos os tipos de ocorrência únicos para este número de ocorrência
    tipos_de_ocorrencia_para_este_num = df_filtrado_por_num['Tipo_de_Ocorrencia'].unique()

    # Verifica se há mais de um tipo de ocorrência único para este número de ocorrência
    # Se houver mais de um, significa que o Tipo_de_Ocorrencia se repete (ou seja, varia)
    qtd_tipos_unicos = len(df_filtrado_por_num)
    tipo_de_ocorrencia_varia_nesta_repeticao = len(tipos_de_ocorrencia_para_este_num) > 1

    # Adiciona o resultado à lista: [número da ocorrência, lista de tipos únicos, se o tipo varia]
    lista_resultados_tipo_ocorrencia.append(
        [num_ocorrencia, tipos_de_ocorrencia_para_este_num.tolist(), tipo_de_ocorrencia_varia_nesta_repeticao, qtd_tipos_unicos]
    )

# Converte a lista de resultados em um array NumPy
# Usamos dtype=object para permitir que as listas de tipos de ocorrência tenham tamanhos variáveis
array_verificacao_tipos = np.array(lista_resultados_tipo_ocorrencia, dtype=object)

print("=== Verificação de 'Tipo_de_Ocorrencia' para 'Numero_da_Ocorrencia' repetidos ===")
print("Colunas: [Número da Ocorrência, Tipos Únicos de Ocorrência, Tipo de Ocorrência varia?]")
print(array_verificacao_tipos)

print('-' * 50)
print('Ocorrencias Verdadeiramente Dupllicadas')



Executando as exclusões com combinação entre [Numero_da_Ocorrencia] & [Tipo_de_Ocorrencia]

In [ ]:
#Analise Antes da Execução do código
print(f"Quantidade de linhas antes da remoção de duplicatas: {len(df_tratado)}")
display(df_tratado[df_tratado['Numero_da_Ocorrencia'] == 36228])

# Remove duplicatas com base na combinação de 'Numero_da_Ocorrencia' e 'Tipo_de_Ocorrencia'
df_tratado.drop_duplicates(subset=['Numero_da_Ocorrencia', 'Tipo_de_Ocorrencia'], inplace=True)

#Analise Após a Execução do código
print(f"Quantidade de linhas após a remoção de duplicatas: {len(df_tratado)}")
display(df_tratado[df_tratado['Numero_da_Ocorrencia'] == 36228])

## Verificando as novas dimensões do DataFrame

In [ ]:
tamanho_df_final = len(df)
print("Quantidade de Registros:")
print(f'Inicial do DataFrame: {tamanho_df_inicio}')
print(f"Final do DataFrame: {tamanho_df_final}")
print(f'Diferença: {tamanho_df_final - tamanho_df_inicio}')

memoria_df_final = df.memory_usage(deep=True).sum()
print("Quantidade de Memória:")
print(f"Inicial do DataFrame em bytes: {memoria_df_inicio}")
print(f"Final do DataFrame em bytes: {memoria_df_final}")
print(f'Diferença: {memoria_df_final - memoria_df_inicio}')

## Analise Estatística Descritiva Geral

In [ ]:
df.describe()